# 07 — PE-G14 Score Matrix (Upgrade 1: Round 4 của iterative ensemble)

Load PE-G14 embeddings từ `01a` rồi compute Q×G similarity matrix theo format chuẩn (cùng schema với `scores_uit.pt`, `scores_blip2.pt`, `scores_clip.pt`).

Đây là Round 4 của 4-vòng ensemble (UIT → BLIP-2 → CLIP → **PE-G14**). PE-G14 1.8B params là model mạnh nhất → để round cuối kéo mạnh hơn (w_4 = 0.85, thấp nhất).

**Inputs:**
- `features/pe_g14/{gallery,queries,val,val_zs}.npz` từ 01a
- `features/pe_g14/{val_text,val_zs_text}.npz`

**Outputs:**
- `features/pe_g14/scores_pe.pt`
- `features/pe_g14/scores_val.pt`, `scores_val_zs.pt`

**ETA:** ~5 phút.

In [ ]:
from pathlib import Path
import os, sys, warnings
import numpy as np
import torch, torch.nn.functional as F
warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path('.').resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from aic_colab_utils import (
    setup_aic2026_environment, select_a100_device,
    sync_chunk_to_drive, wait_for_pending_syncs,
)

PATHS = setup_aic2026_environment()
LOCAL_ROOT = PATHS['local_root']; OUTPUT_ROOT = PATHS['output_root']
DRIVE_OUTPUT_ROOT = PATHS['drive_output_root']
PE_FEAT_DIR = OUTPUT_ROOT / 'features' / 'pe_g14'
device = select_a100_device()

def _score(q_emb, g_emb):
    Q = F.normalize(torch.from_numpy(q_emb.astype('float32')).to(device), dim=-1)
    G = F.normalize(torch.from_numpy(g_emb.astype('float32')).to(device), dim=-1)
    return (Q @ G.T).half().cpu()

z_g = np.load(PE_FEAT_DIR / 'gallery.npz')
z_q = np.load(PE_FEAT_DIR / 'queries.npz')
S = _score(z_q['embeddings'], z_g['embeddings'])
torch.save({'scores': S, 'query_ids': z_q['ids'], 'gallery_ids': z_g['ids']},
           PE_FEAT_DIR / 'scores_pe.pt')
print('scores_pe.pt:', tuple(S.shape))

for name in ('val', 'val_zs'):
    ip = PE_FEAT_DIR / f'{name}.npz'
    tp = PE_FEAT_DIR / f'{name}_text.npz'
    if not (ip.exists() and tp.exists()):
        print(f'⚠️  Skip {name}: missing {ip.name} or {tp.name}')
        continue
    zi = np.load(ip); zt = np.load(tp)
    Sv = _score(zt['embeddings'], zi['embeddings'])
    torch.save({'scores': Sv, 'query_ids': zt['ids'], 'gallery_ids': zi['ids']},
               PE_FEAT_DIR / f'scores_{name}.pt')
    print(f'scores_{name}.pt:', tuple(Sv.shape))

for f in [PE_FEAT_DIR / 'scores_pe.pt', PE_FEAT_DIR / 'scores_val.pt', PE_FEAT_DIR / 'scores_val_zs.pt']:
    if f.exists():
        sync_chunk_to_drive(f, LOCAL_ROOT, DRIVE_OUTPUT_ROOT, background=True)
wait_for_pending_syncs()
print('Done.')